# Joins in SLayer

Joins connect models so that dimensions and measures from one model are accessible when querying another. SLayer only supports **LEFT JOINs** — the kind most commonly used for data enrichment.

If you think of left joins as directed edges of a graph whose vertices are models, then the dimensions, measures, and filters in a model have access to columns from any model reachable in the join graph.

This notebook illustrates every aspect of joins with working code.

See also: [Models — Joins](../../concepts/models.md#joins) | [Queries — Cross-Model Measures](../../concepts/queries.md#cross-model-measures) | [Ingestion — Diamond Joins](../../concepts/ingestion.md#diamond-joins)

**Prerequisites:** `pip install motley-slayer[examples]`

In [1]:
import os
import sys

sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", ".."))
sys.path.insert(0, os.path.join(os.getcwd(), "..", "jaffle_data"))

from setup_jaffle import ensure_jaffle_shop

from slayer.core.query import SlayerQuery

engine, storage, models = ensure_jaffle_shop()

/home/james/miniconda3/envs/motley3.11/lib/python3.11/site-packages/sqlglot/tokens.py:14: UserWarning: sqlglot[rs] is deprecated and no longer compatible with sqlglot. Please use sqlglotc instead for faster parsing: pip install sqlglot[c]
  warnings.warn(


## Basic Joins

A join is defined by a **target model** and a list of **join pairs** — column pairs to join on. The `orders` model has two joins, auto-generated from its foreign keys to `customers` and `stores`:

In [2]:
orders_model = next(m for m in models if m.name == "orders")

print("orders model joins:")
for j in orders_model.joins:
    print(f"  -> {j.target_model}  ON {j.join_pairs}")

# Query using a joined dimension
result = engine.execute(query=SlayerQuery(
    source_model="orders",
    fields=[{"formula": "count"}, {"formula": "order_total_sum"}],
    dimensions=[{"name": "customers.name"}],
    order=[{"column": {"name": "order_total_sum"}, "direction": "desc"}],
    limit=5,
))

print("\nTop 5 customers by revenue:")
for row in result.data:
    print(f"  {row['orders.customers.name']}: {row['orders.count']} orders, ${row['orders.order_total_sum']:,.2f}")

orders model joins:
  -> customers  ON [['customer_id', 'id']]
  -> stores  ON [['store_id', 'id']]

Top 5 customers by revenue:
  Rachel Johnson: 708 orders, $8,623.73
  Anna Jones: 560 orders, $6,824.82
  Eric Smith: 567 orders, $6,757.92
  Keith Martinez: 536 orders, $6,575.62
  Nichole Haynes: 514 orders, $6,399.76


## Referencing Joined Models in Queries (Dot Syntax)

SLayer uses **dot syntax** to reference dimensions and measures from joined models:

- 1-hop: `customers.name` (orders -> customers)
- Multi-hop: `orders.customers.name` (order_items -> orders -> customers)

The full path avoids ambiguity when there are multiple ways to reach a model.

In [3]:
# 1-hop: orders -> stores
result = engine.execute(query=SlayerQuery(
    source_model="orders",
    fields=[{"formula": "count"}],
    dimensions=[{"name": "stores.name"}],
))

print("1-hop (orders -> stores):")
for row in result.data:
    print(f"  {row['orders.stores.name']}: {row['orders.count']} orders")

1-hop (orders -> stores):
  San Francisco: 28453 orders
  New Orleans: 4293 orders
  Chicago: 34997 orders
  Brooklyn: 80349 orders
  Philadelphia: 58293 orders


In [4]:
# Multi-hop: order_items -> orders -> customers
result = engine.execute(query=SlayerQuery(
    source_model="order_items",
    fields=[{"formula": "count"}, {"formula": "quantity_sum"}],
    dimensions=[{"name": "orders.customers.name"}],
    order=[{"column": {"name": "quantity_sum"}, "direction": "desc"}],
    limit=5,
))

print("Multi-hop (order_items -> orders -> customers):")
for row in result.data:
    print(f"  {row['order_items.orders.customers.name']}: {row['order_items.count']} line items, {row['order_items.quantity_sum']} qty")

Multi-hop (order_items -> orders -> customers):
  Rachel Johnson: 1077 line items, 1093 qty
  Eric Smith: 872 line items, 888 qty
  Anna Jones: 837 line items, 837 qty
  Keith Martinez: 809 line items, 809 qty
  Adam Little: 782 line items, 782 qty


## Referencing Joined Models in SQL Snippets

When defining dimensions and measures at the model level (in YAML or Python), their `sql` fields use **double underscores** instead of dots for joined table aliases. This is because dots aren't valid in SQL identifiers.

| Context | Syntax | Example |
|---------|--------|---------|
| Query dimensions/measures | dots | `orders.customers.name` |
| Model SQL expressions | `__` aliases | `orders__customers.name` |

SLayer substitutes the `__` aliases for correct table aliases at query time.

In [5]:
oi_model = next(m for m in models if m.name == "order_items")

# Show the name vs sql for multi-hop dimensions
print(f"{'Dimension name (dots)':<30} {'SQL expression (__ alias)':<30}")
print("-" * 62)
for dim in oi_model.dimensions:
    if "__" in (dim.sql or ""):
        print(f"{dim.name:<30} {dim.sql:<30}")

Dimension name (dots)          SQL expression (__ alias)     
--------------------------------------------------------------
orders.customers.id            orders__customers.id          
orders.customers.name          orders__customers.name        
orders.stores.id               orders__stores.id             
orders.stores.name             orders__stores.name           
orders.stores.opened_at        orders__stores.opened_at      
orders.stores.tax_rate         orders__stores.tax_rate       


## Auto-Ingesting Schemas

When auto-ingesting a database, SLayer introspects FK constraints and creates `ModelJoin` objects automatically. It follows FK chains transitively — if `order_items` -> `orders` -> `customers`, then the `order_items` model gets joins to all three tables.

Multi-hop join pairs use path-qualified source columns to indicate which already-joined table the FK comes from:

In [6]:
import sqlalchemy as sa

from setup_jaffle import DB_PATH
from slayer.core.models import DatasourceConfig
from slayer.engine.ingestion import _build_fk_graph

ds = DatasourceConfig(name="jaffle_shop", type="duckdb", database=DB_PATH)
sa_engine = sa.create_engine(ds.resolve_env_vars().get_connection_string())
inspector = sa.inspect(sa_engine)
fk_graph = _build_fk_graph(inspector=inspector, table_names=inspector.get_table_names(), schema=None)
sa_engine.dispose()

print("FK graph:")
for table in sorted(fk_graph):
    print(f"  {table} -> {sorted(fk_graph[table])}")

print("\norder_items joins (auto-generated):")
for j in oi_model.joins:
    src = j.join_pairs[0][0]
    tgt = j.join_pairs[0][1]
    hop = "multi-hop" if "." in src else "direct"
    print(f"  -> {j.target_model:<12} ON {src:<20} = {tgt:<5} ({hop})")

FK graph:
  order_items -> ['orders', 'products']
  orders -> ['customers', 'stores']
  supplies -> ['products']
  tweets -> ['customers']

order_items joins (auto-generated):
  -> orders       ON order_id             = id    (direct)
  -> products     ON sku                  = sku   (direct)
  -> customers    ON orders.customer_id   = id    (multi-hop)
  -> stores       ON orders.store_id      = id    (multi-hop)


## Diamond Joins

A **diamond join** occurs when the same table is reachable via multiple FK paths. For example:

```
orders -> customers -> regions
orders -> warehouses -> regions
```

SLayer treats each path as a separate copy of the target table, using **path-based aliases** to disambiguate:
- `customers.regions.name` -> table alias `customers__regions`  
- `warehouses.regions.name` -> table alias `warehouses__regions`

The Jaffle Shop schema doesn't have a natural diamond, so let's construct one:

In [7]:
from joins_utils import setup_diamond_example

diamond_engine, diamond_storage, diamond_models, diamond_db_path, diamond_work_dir = setup_diamond_example()

diamond_orders = next(m for m in diamond_models if m.name == "orders")

print("Diamond orders model joins:")
for j in diamond_orders.joins:
    src = j.join_pairs[0][0]
    tgt = j.join_pairs[0][1]
    print(f"  -> {j.target_model:<12} ON {src:<25} = {tgt}")

print("\nDimensions with path aliases:")
for dim in diamond_orders.dimensions:
    if "regions" in dim.name:
        print(f"  name: {dim.name:<35} sql: {dim.sql}")

Diamond orders model joins:
  -> customers    ON customer_id               = id
  -> warehouses   ON warehouse_id              = id
  -> regions      ON customers.region_id       = id
  -> regions      ON warehouses.region_id      = id

Dimensions with path aliases:
  name: customers.regions.id                sql: customers__regions.id
  name: customers.regions.name              sql: customers__regions.name
  name: warehouses.regions.id               sql: warehouses__regions.id
  name: warehouses.regions.name             sql: warehouses__regions.name


In [8]:
# Query both paths simultaneously
result = diamond_engine.execute(query=SlayerQuery(
    source_model="orders",
    fields=[{"formula": "count"}, {"formula": "amount_sum"}],
    dimensions=[
        {"name": "customers.regions.name"},
        {"name": "warehouses.regions.name"},
    ],
))

print(f"{'Customer Region':<18} {'Warehouse Region':<18} {'Orders':>7} {'Amount':>10}")
print("-" * 56)
for row in result.data:
    cr = row["orders.customers.regions.name"]
    wr = row["orders.warehouses.regions.name"]
    print(f"{cr:<18} {wr:<18} {row['orders.count']:>7} ${row['orders.amount_sum']:>9,.2f}")

Customer Region    Warehouse Region    Orders     Amount
--------------------------------------------------------
East               Central                  2 $   375.00
East               West                     2 $   600.00
Central            West                     1 $   225.00
Central            Central                  1 $   150.00
West               East                     3 $   750.00
West               West                     1 $   175.00


### Recombining Diamond Joins with Filters

By default, each path to the same table produces independent copies. If you want to enforce that the two paths resolve to the same row (re-creating a true diamond), add a filter equating the two paths.

Since this filter equates two SQL-level table aliases (`customers__regions` vs `warehouses__regions`), it belongs on the **model** — added via `ModelExtension` at query time. Model filters use the `__` alias syntax for multi-hop join paths (see [SQL vs DSL](../sql_vs_dsl/sql_vs_dsl.md)):

In [ ]:
# Only keep orders where customer and warehouse are in the same region.
# The filter equates two SQL aliases, so it goes on the model via ModelExtension.
result = diamond_engine.execute(query=SlayerQuery(
    source_model={
        "source_name": "orders",
        "filters": ["customers__regions.name = warehouses__regions.name"],
    },
    fields=[{"formula": "count"}, {"formula": "amount_sum"}],
    dimensions=[{"name": "customers.regions.name"}],
))

print("Orders where customer and warehouse are in the same region:")
for row in result.data:
    print(f"  {row['orders.customers.regions.name']}: {row['orders.count']} orders, ${row['orders.amount_sum']:,.2f}")

## Dynamic Joins (ModelExtension)

Joins can be added at query time via `ModelExtension`, without modifying the stored model. This is useful for:
- Ad-hoc joins to lookup tables
- Joining to models created dynamically from queries
- Adding context-specific enrichment

More on this in the upcoming post (and companion notebook) on multistage queries in Slayer.

## Summary

SLayer's join system provides:

| Feature | Description |
|---------|-------------|
| **LEFT JOINs** | The only join type, used for data enrichment |
| **Dot syntax** | `customers.name`, `orders.customers.regions.name` in queries |
| **`__` aliases** | `orders__customers.name` in model SQL definitions |
| **Auto-ingestion** | FK constraints become `ModelJoin` objects automatically |
| **Diamond joins** | Same table via multiple paths gets separate path-based aliases |
| **Dynamic joins** | `ModelExtension` adds joins at query time without modifying models |

See the [Models docs](../../concepts/models.md#joins) and [Ingestion docs](../../concepts/ingestion.md) for the full reference.